# TinyML3 · Explorador de grabaciones IMU

Selecciona participantes, actividades, tomas y archivos para **mostrar las señales originales**.
Cada grabación genera una figura con dos paneles: acelerómetro X/Y/Z (g) y giroscopio X/Y/Z (°/s).
No se escriben CSV ni archivos de imagen y no se modifican las grabaciones.

## 1. Dependencias

Necesita `numpy`, `pandas`, `matplotlib`
e `ipywidgets`. Si faltan, instálalos desde una terminal con:

```bash
pip install numpy pandas matplotlib ipywidgets ipykernel
```

Ejecuta las celdas en orden. Los gráficos se generan al pulsar **Graficar selección**.
Para selección múltiple utiliza Ctrl/Cmd + clic o Mayús + clic.


In [1]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams.update({"figure.dpi": 100, "font.size": 10})


## 2. Localizar el proyecto

La búsqueda parte del directorio de trabajo del kernel y recorre sus carpetas superiores.
Funciona al iniciar Jupyter desde `TinyML3/` o `Preprocessing/`, sin rutas absolutas fijas.
Si Jupyter arrancó en otro lugar, cambia solamente `PROJECT_ROOT` por la ruta a tu `TinyML3`.


In [2]:
PROJECT_ROOT = None  # Ejemplo: Path("/ruta/a/TinyML3")
DATA_RELATIVE = Path("Data_adquisition") / "datos"

def localizar_proyecto(root=None):
    if root is not None:
        candidate = Path(root).expanduser().resolve()
        if not (candidate / DATA_RELATIVE).is_dir():
            raise FileNotFoundError(f"No existe {candidate / DATA_RELATIVE}")
        return candidate
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / DATA_RELATIVE).is_dir():
            return candidate
    raise FileNotFoundError(
        "No encuentro Data_adquisition/datos desde el directorio del kernel. "
        "Inicia Jupyter dentro de TinyML3 o configura PROJECT_ROOT. "
        f"Directorio actual: {cwd}"
    )

ROOT = localizar_proyecto(PROJECT_ROOT)
DATA_DIR = ROOT / DATA_RELATIVE
print(f"Proyecto: {ROOT}")
print(f"Grabaciones: {DATA_DIR}")


Proyecto: /home/jrgels/Documents/Maestría/Version 2/TinyML3
Grabaciones: /home/jrgels/Documents/Maestría/Version 2/TinyML3/Data_adquisition/datos


## 3. Inventario de CSV

Se buscan recursivamente nombres `P<número>_<actividad>_<toma>.csv`; la actividad puede
contener guiones bajos, como `subir_escaleras`. Los JSON no se utilizan.
No se presupone que haya tres tomas por actividad ni que la numeración sea consecutiva.
La tabla cuenta los archivos que realmente existen, sin inventar tomas faltantes.


In [3]:
NAME_PATTERN = re.compile(r"^(P[0-9]+)_(.+)_([0-9]+)$", re.IGNORECASE)

def crear_catalogo(data_dir):
    records, ignored = [], []
    for path in sorted(Path(data_dir).rglob("*")):
        if not path.is_file() or path.suffix.lower() != ".csv":
            continue
        match = NAME_PATTERN.fullmatch(path.stem)
        if match is None:
            ignored.append(path.relative_to(data_dir).as_posix())
            continue
        participant, activity, take = match.groups()
        records.append({
            "participante": participant.upper(),
            "actividad": activity.lower(),
            "toma": take.zfill(3),
            "archivo": path.relative_to(data_dir).as_posix(),
            "ruta": path,
        })
    if not records:
        raise FileNotFoundError(
            f"No hay CSV con nombres P<n>_<actividad>_<toma>.csv en {data_dir}"
        )
    catalog = pd.DataFrame(records)
    # Orden numérico: P2 antes que P10, toma 003 antes que 010.
    catalog = catalog.sort_values(
        ["participante", "actividad", "toma", "archivo"],
        key=lambda s: s.str[1:].astype(int) if s.name == "participante"
        else s.astype(int) if s.name == "toma" else s,
    ).reset_index(drop=True)
    return catalog, ignored

catalogo, ignorados = crear_catalogo(DATA_DIR)
print(f"{len(catalogo)} grabaciones encontradas.")
display(catalogo.groupby(["participante", "actividad"]).size().unstack(fill_value=0))
if ignorados:
    print("CSV omitidos porque su nombre no sigue el patrón esperado:")
    print("\n".join(ignorados))


44 grabaciones encontradas.


actividad,bajar_escaleras,caminar,correr,reposo,subir_escaleras
participante,,,,,
P1,2,3,3,3,4
P2,3,3,3,3,3
P3,3,3,3,2,3


## 4. Lectura y gráficas

El ejemplo `P3_caminar_001.csv` contiene las columnas
`t_s, ax_g, ay_g, az_g, gx_dps, gy_dps, gz_dps`: 3,000 muestras, de 0.00 a 29.99 s,
con paso de 0.01 s (100 Hz nominales). El eje horizontal usa **el tiempo del CSV**.

Cada archivo se valida antes de graficarlo. Se avisa si el muestreo es irregular y se
rechazan columnas faltantes, valores no numéricos/no finitos o tiempos que no aumentan.
No se interpolan, ordenan ni eliminan filas silenciosamente.

Se muestran datos crudos para inspección; la normalización Z-score del modelo se hará
por separado, usando las estadísticas del conjunto de entrenamiento.


In [4]:
TIME = "t_s"
ACC = ["ax_g", "ay_g", "az_g"]
GYRO = ["gx_dps", "gy_dps", "gz_dps"]
REQUIRED = [TIME, *ACC, *GYRO]

def leer_grabacion(path):
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = df.columns.str.strip()
    missing = [column for column in REQUIRED if column not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas: {', '.join(missing)}")
    if df.columns.duplicated().any():
        raise ValueError("Hay nombres de columna duplicados después de quitar espacios.")
    if len(df) < 2:
        raise ValueError("Se necesitan al menos dos muestras para representar el tiempo.")
    data = df[REQUIRED].apply(pd.to_numeric, errors="raise")
    if not np.isfinite(data.to_numpy(dtype=float)).all():
        raise ValueError("Hay valores vacíos, NaN o infinitos en el tiempo o los canales IMU.")
    dt = np.diff(data[TIME].to_numpy(dtype=float))
    if np.any(dt <= 0):
        raise ValueError("t_s debe aumentar estrictamente; hay tiempos repetidos o retrocesos.")
    step = float(np.median(dt))
    irregular = not np.allclose(dt, step, rtol=0.05, atol=1e-9)
    if irregular:
        warnings.warn(
            f"{Path(path).name}: intervalos de tiempo irregulares (>5% del paso mediano). "
            "Se conserva el tiempo original.", stacklevel=2
        )
    summary = {
        "muestras": len(data),
        "intervalo_s": float(data[TIME].iloc[-1] - data[TIME].iloc[0]),
        "frecuencia_mediana_hz": 1.0 / step,
        "tiempo_irregular": irregular,
    }
    return data, summary

def graficar_grabacion(record):
    data, summary = leer_grabacion(record["ruta"])
    fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True, layout="constrained")
    try:
        colors = ["#0072B2", "#D55E00", "#009E73"]
        for ax, columns, title, unit in (
            (axes[0], ACC, "Acelerómetro", "Aceleración (g)"),
            (axes[1], GYRO, "Giroscopio", "Velocidad angular (°/s)"),
        ):
            for column, axis, color in zip(columns, "XYZ", colors):
                ax.plot(data[TIME], data[column], label=axis, color=color, linewidth=0.8)
            ax.set(title=title, ylabel=unit)
            ax.legend(loc="upper right", ncol=3)
            ax.grid(True, alpha=0.25)
        axes[1].set_xlabel("Tiempo del CSV (s)")
        fig.suptitle(
            f"{record['participante']} · {record['actividad']} · toma {record['toma']}\n"
            f"{record['archivo']}\n"
            f"{summary['muestras']:,} muestras · intervalo {summary['intervalo_s']:.2f} s · "
            f"1/mediana(Δt) ≈ {summary['frecuencia_mediana_hz']:.2f} Hz"
        )
        display(fig)
    finally:
        # Liberar la figura después de mostrarla evita duplicados y acumulación en memoria.
        plt.close(fig)
    return summary


## 5. Seleccionar y visualizar

1. Filtra por uno o varios participantes, actividades y tomas. Seleccionar todo en un
   filtro significa que ese criterio no restringe los resultados; dejarlo vacío da cero coincidencias.
2. Elige los archivos en la lista o pulsa **Seleccionar todos** para incluir todos los filtrados.
   Inicialmente se selecciona solo el primero para evitar generar decenas de figuras por accidente.
3. Pulsa **Graficar selección**. Se muestran todas las grabaciones seleccionadas y un resumen;
   si un archivo falla, se informa y se continúa con los demás.

Al cambiar filtros revisa la lista de archivos seleccionados. Para ver todo el conjunto,
deja todos los valores de los tres filtros, pulsa **Seleccionar todos** y luego **Graficar selección**.
Tras añadir CSV al proyecto, vuelve a ejecutar desde la sección 3.


In [5]:
def filtrar_catalogo(catalog, participantes, actividades, tomas):
    return catalog.loc[
        catalog["participante"].isin(participantes)
        & catalog["actividad"].isin(actividades)
        & catalog["toma"].isin(tomas)
    ]

def selector(label, values):
    values = tuple(values)
    return widgets.SelectMultiple(
        options=values, value=values, description=label,
        rows=min(8, max(3, len(values))),
        layout=widgets.Layout(width="32%"), style={"description_width": "initial"}
    )

participantes = selector("Participante", catalogo["participante"].unique())
actividades = selector("Actividad", sorted(catalogo["actividad"].unique()))
tomas = selector("Toma", sorted(catalogo["toma"].unique(), key=int))
archivos = widgets.SelectMultiple(
    description="Archivos", rows=12,
    layout=widgets.Layout(width="100%"), style={"description_width": "initial"}
)
estado = widgets.Label()
todos = widgets.Button(description="Seleccionar todos", icon="check")
ninguno = widgets.Button(description="Quitar selección")
graficar = widgets.Button(description="Graficar selección", button_style="primary", icon="line-chart")
salida = widgets.Output(layout={"width": "100%"})

def actualizar_archivos(change=None):
    previous = set(archivos.value)
    filtered = filtrar_catalogo(catalogo, participantes.value, actividades.value, tomas.value)
    options = tuple(filtered["archivo"])
    archivos.options = options
    kept = tuple(name for name in options if name in previous)
    archivos.value = kept if kept else options[:1]
    actualizar_estado()

def actualizar_estado(change=None):
    estado.value = f"{len(archivos.options)} archivos filtrados · {len(archivos.value)} seleccionados"
    graficar.disabled = not bool(archivos.value)

def seleccionar_todos(button):
    archivos.value = tuple(archivos.options)

def quitar_seleccion(button):
    archivos.value = ()

def mostrar_seleccion(button):
    selection = tuple(archivos.value)
    with salida:
        clear_output(wait=True)
        if not selection:
            print("Selecciona al menos un archivo.")
            return
        graficar.disabled = True
        try:
            summaries, errors = [], []
            indexed = catalogo.set_index("archivo", drop=False)
            for filename in selection:
                record = indexed.loc[filename]
                try:
                    summary = graficar_grabacion(record)
                    summaries.append({"archivo": filename, **summary})
                except (OSError, ValueError, pd.errors.ParserError) as exc:
                    errors.append({"archivo": filename, "error": str(exc)})
                    print(f"No se pudo graficar {filename}: {exc}")
            print(f"Graficados: {len(summaries)} · Errores: {len(errors)}")
            if summaries:
                display(pd.DataFrame(summaries))
            if errors:
                display(pd.DataFrame(errors))
        finally:
            actualizar_estado()

for control in (participantes, actividades, tomas):
    control.observe(actualizar_archivos, names="value")
archivos.observe(actualizar_estado, names="value")
todos.on_click(seleccionar_todos)
ninguno.on_click(quitar_seleccion)
graficar.on_click(mostrar_seleccion)
actualizar_archivos()
display(widgets.VBox([
    widgets.HBox([participantes, actividades, tomas]), archivos, estado,
    widgets.HBox([todos, ninguno, graficar]), salida
]))


## 6. Alternativa por código

Si el entorno no muestra los controles de widgets, puedes usar las funciones de arriba
directamente después de ejecutar las secciones 1–4. Quita los comentarios para ejecutar
alguno de estos ejemplos. Usa rutas relativas a `Data_adquisition/datos` para distinguir archivos.


In [6]:
# Ejemplo A: participante P3, caminar, toma 001.
# seleccion = filtrar_catalogo(catalogo, ["P3"], ["caminar"], ["001"])
# for _, grabacion in seleccion.iterrows():
#     graficar_grabacion(grabacion)

# Ejemplo B: una lista explícita de archivos.
# elegidos = ["Persona3/Caminar/P3_caminar_001.csv"]
# for nombre in elegidos:
#     coincidencias = catalogo.loc[catalogo["archivo"].eq(nombre)]
#     if coincidencias.empty:
#         print(f"No encontrado: {nombre}")
#     else:
#         graficar_grabacion(coincidencias.iloc[0])


**Sobre guardar las gráficas:** este notebook no llama a `savefig` ni crea imágenes en disco.
Jupyter puede incluir las salidas visibles dentro del propio `.ipynb` al guardarlo.
Si tampoco quieres conservarlas ahí, usa «Clear All Outputs / Borrar todas las salidas» antes
de guardar o compartir el notebook. La copia entregada tiene sus salidas vacías.
